# Thai Math VQA — Colab run (clean / restart-proof)

Every run cell uses `cd /content/thai-math-vqa && ...` so a runtime restart can't break paths.

**Do this first:** `Runtime → Change runtime type → GPU (A100 or L4) → Save`.

Then run the cells **top to bottom** (click the ▶️ on each, wait for the green ✓).

## 1. Check the GPU

In [ ]:
!nvidia-smi


## 2. Install (~3 min)
If Colab shows a **RESTART RUNTIME** button, click it, then continue from cell 3 (do not re-run this).

In [ ]:
%pip install -q -U vllm "transformers>=4.49.0" qwen-vl-utils accelerate pythainlp kaggle pillow pandas
print('installed')


## 3. Get the code (public repo)

In [ ]:
!git clone https://github.com/mingrath/thai-math-vqa.git /content/thai-math-vqa 2>/dev/null || (cd /content/thai-math-vqa && git pull --ff-only)
!ls /content/thai-math-vqa


## 4. Kaggle key
Click **Choose Files** and upload your `kaggle.json` (Kaggle → Settings → Create New API Token).

In [ ]:
from google.colab import files
import os, glob, shutil
files.upload()
os.makedirs('/root/.kaggle', exist_ok=True)
src = [f for f in glob.glob('*.json') if 'kaggle' in f.lower()]
shutil.copy(src[0] if src else 'kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('kaggle creds set from', src[0] if src else 'kaggle.json')


## 5. Download the data
Reuses data if already present. Needs competition rules accepted on Kaggle (else 403).

In [ ]:
!cd /content/thai-math-vqa && (ls data/images >/dev/null 2>&1 && echo 'data already present' || (kaggle competitions download -c super-ai-engineer-ss-6-individual-test-thai-math-vqa-challen -p data/ && cd data && unzip -q -o '*.zip' && rm -f *.zip))
!echo '--- data ---' && ls /content/thai-math-vqa/data


## 6. Sanity check (data + normalization)

In [ ]:
!cd /content/thai-math-vqa && python -m src.data data && echo '--- normalize self-test ---' && python src/normalize.py


## 7. Smoke test (16 images)
First run downloads the model (~16GB, a few minutes). Success = an accuracy number prints.

In [ ]:
!cd /content/thai-math-vqa && python run_pipeline.py --mode eval --limit 16 --n 1


## 8. Full practice score on train (n=8)
Your offline estimate of the leaderboard score. Tune here before spending a submission.

In [ ]:
!cd /content/thai-math-vqa && python run_pipeline.py --mode eval --n 8 --save-eval eval_train.csv


## 9. Predict test → submission.csv

In [ ]:
!cd /content/thai-math-vqa && python run_pipeline.py --mode predict --n 8 --out submissions/sub.csv && echo DONE && head submissions/sub.csv


## 10. Submit to Kaggle
Limit: 5 submissions total — submit your best.

In [ ]:
!cd /content/thai-math-vqa && kaggle competitions submit -c super-ai-engineer-ss-6-individual-test-thai-math-vqa-challen -f submissions/sub.csv -m 'qwen2.5-vl-7b self-consistency n=8'


## Raise the score (edit the commands above)
- **More samples:** change `--n 8` to `--n 16` (biggest single lever).
- **Better model:** add `--model Qwen/Qwen3-VL-8B-Instruct` (stronger Thai+math).
- **Higher resolution:** add `--max-pixels 1605632` (helps OCR; lower if you OOM).
- **Out of memory?** add `--max-pixels 802816` or `--n 4`.
- Inspect `eval_train.csv` to see misses, then improve `src/prompts.py` / `src/normalize.py`.